# 📈 VN Stock Backtester — RSI Reversal + Walk-Forward
### Dữ liệu thật từ TCBS/SSI qua vnstock · 3 năm lịch sử HOSE/HNX

---
**Cách dùng:** Chạy từng ô theo thứ tự từ trên xuống (Shift+Enter hoặc nhấn ▶)

> ⚠️ **Miễn trách nhiệm:** Kết quả backtest là lịch sử, không đảm bảo lợi nhuận tương lai. Không phải tư vấn đầu tư.


## ⚙️ BƯỚC 1 — Cài đặt thư viện

In [ ]:
# Cài vnstock và các thư viện cần thiết
!pip install vnstock==3.2.0 pandas numpy matplotlib seaborn scipy -q
print('✅ Cài đặt xong!')

## 📦 BƯỚC 2 — Import & Hàm tiện ích

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
from datetime import datetime, timedelta
from scipy import stats
import itertools

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10
sns.set_style('darkgrid')

# ── Màu sắc ──────────────────────────────────────────────────────────────────
CLR = dict(up='#26a69a', down='#ef5350', neutral='#90a4ae',
           bg='#1e1e2e', surface='#2a2a3e', accent='#7c4dff',
           gold='#ffd600', green='#00e676', red='#ff1744')

# ── Hàm tải dữ liệu ──────────────────────────────────────────────────────────
def load_history(symbol: str, years: int = 3) -> pd.DataFrame:
    """
    Tải dữ liệu lịch sử từ TCBS qua vnstock.
    Trả về DataFrame có cột: date, open, high, low, close, volume
    """
    from vnstock import Vnstock
    end   = datetime.now().strftime('%Y-%m-%d')
    start = (datetime.now() - timedelta(days=years*365+30)).strftime('%Y-%m-%d')
    try:
        stk = Vnstock().stock(symbol=symbol, source='TCBS')
        df  = stk.quote.history(start=start, end=end, interval='1D')
        if df is None or df.empty:
            raise ValueError('Không có dữ liệu')
        df = df.rename(columns={
            'time':'date','open':'open','high':'high',
            'low':'low','close':'close','volume':'volume'
        })
        df['date']   = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)
        # TCBS trả đơn vị nghìn đồng → đổi sang đồng
        for col in ['open','high','low','close']:
            if df[col].max() < 10000:
                df[col] = df[col] * 1000
        df['symbol'] = symbol
        print(f'  ✅ {symbol}: {len(df)} phiên ({df["date"].min().date()} → {df["date"].max().date()})')
        return df
    except Exception as e:
        print(f'  ❌ {symbol}: {e}')
        return pd.DataFrame()

# ── Chỉ báo kỹ thuật ─────────────────────────────────────────────────────────
def add_indicators(df: pd.DataFrame, rsi_period=14) -> pd.DataFrame:
    df = df.copy()
    close = df['close']

    # RSI
    delta = close.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_g = gain.ewm(com=rsi_period-1, min_periods=rsi_period).mean()
    avg_l = loss.ewm(com=rsi_period-1, min_periods=rsi_period).mean()
    rs    = avg_g / avg_l.replace(0, np.nan)
    df['rsi'] = 100 - 100 / (1 + rs)

    # EMA & SMA
    df['ema20'] = close.ewm(span=20, min_periods=20).mean()
    df['sma50'] = close.rolling(50, min_periods=50).mean()

    # MACD
    ema12       = close.ewm(span=12, min_periods=12).mean()
    ema26       = close.ewm(span=26, min_periods=26).mean()
    df['macd']  = ema12 - ema26
    df['macd_signal'] = df['macd'].ewm(span=9, min_periods=9).mean()
    df['macd_hist']   = df['macd'] - df['macd_signal']

    # Bollinger Bands
    mid           = close.rolling(20, min_periods=20).mean()
    std           = close.rolling(20, min_periods=20).std()
    df['bb_mid']  = mid
    df['bb_upper']= mid + 2*std
    df['bb_lower']= mid - 2*std

    # Volume MA
    df['vol_ma20'] = df['volume'].rolling(20).mean()
    df['vol_ratio']= df['volume'] / df['vol_ma20']

    # ATR (Average True Range) — đo độ biến động
    tr = pd.concat([
        df['high'] - df['low'],
        (df['high'] - close.shift()).abs(),
        (df['low']  - close.shift()).abs()
    ], axis=1).max(axis=1)
    df['atr14'] = tr.rolling(14).mean()

    return df

print('✅ Các hàm đã được load xong!')


## 🎯 BƯỚC 3 — Cấu hình chiến lược RSI

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          CHỈNH SỬA CÁC THAM SỐ TẠI ĐÂY                     ║
# ╚══════════════════════════════════════════════════════════════╝

# Danh sách cổ phiếu muốn test (tối đa 10 để không quá lâu)
SYMBOLS = ['VCB', 'HPG', 'VHM', 'FPT', 'MWG', 'TCB', 'BID', 'VNM', 'CTG', 'GAS']

# Số năm lịch sử (3 năm = dữ liệu thật HOSE/HNX từ TCBS)
YEARS = 3

# Tham số RSI
RSI_PERIOD   = 14
RSI_OVERSOLD = 30    # Ngưỡng quá bán → tín hiệu MUA
RSI_OVERBOUGHT = 70  # Ngưỡng quá mua → tín hiệu BÁN

# Quản lý vốn
INITIAL_CAPITAL = 100_000_000  # 100 triệu VNĐ
RISK_PER_TRADE  = 0.02         # Rủi ro 2% vốn mỗi lệnh
STOP_LOSS_PCT   = 0.05         # Stop loss 5%
TAKE_PROFIT_PCT = 0.10         # Take profit 10%
MAX_HOLD_DAYS   = 20           # Đóng lệnh sau tối đa 20 phiên
COMMISSION      = 0.0015       # Phí giao dịch 0.15% mỗi chiều (thực tế VN)

# Walk-Forward: chia dữ liệu thành bao nhiêu cửa sổ
WF_N_SPLITS     = 6   # 6 cửa sổ thời gian

print('✅ Cấu hình:')
print(f'   Cổ phiếu: {SYMBOLS}')
print(f'   Dữ liệu:  {YEARS} năm lịch sử thực')
print(f'   RSI:      Mua khi RSI < {RSI_OVERSOLD} | Bán khi RSI > {RSI_OVERBOUGHT}')
print(f'   SL/TP:    -{STOP_LOSS_PCT*100:.0f}% / +{TAKE_PROFIT_PCT*100:.0f}%')
print(f'   Phí:      {COMMISSION*100:.2f}% mỗi chiều')


## 📥 BƯỚC 4 — Tải dữ liệu lịch sử thực từ TCBS

In [ ]:
print(f'🔄 Đang tải dữ liệu {YEARS} năm cho {len(SYMBOLS)} cổ phiếu từ TCBS...')
print('   (Mỗi mã mất 3-5 giây, tổng ~30-50 giây)\n')

all_data = {}
failed   = []

for sym in SYMBOLS:
    df = load_history(sym, years=YEARS)
    if not df.empty:
        df = add_indicators(df, rsi_period=RSI_PERIOD)
        all_data[sym] = df
    else:
        failed.append(sym)

print(f'\n📊 Kết quả: {len(all_data)}/{len(SYMBOLS)} mã tải thành công')
if failed:
    print(f'   ❌ Thất bại: {failed} — bỏ qua trong backtest')

# Tổng quan dữ liệu
if all_data:
    sample = list(all_data.values())[0]
    print(f'\n   Ví dụ {list(all_data.keys())[0]}:')
    print(f'   Cột:  {list(sample.columns)}')
    print(f'   Dòng: {len(sample)} phiên giao dịch')
    display(sample[['date','open','high','low','close','volume','rsi','macd']].tail(3))


## 🔬 BƯỚC 5 — Engine Backtest RSI Reversal

In [ ]:
def run_backtest(
    df: pd.DataFrame,
    rsi_oversold   = 30,
    rsi_overbought = 70,
    sl_pct         = 0.05,
    tp_pct         = 0.10,
    max_hold       = 20,
    commission     = 0.0015,
    capital        = 100_000_000,
    risk_per_trade = 0.02,
) -> dict:
    """
    Backtest thuần túy:
    - Mua khi RSI cắt lên từ dưới ngưỡng oversold
    - Đóng khi: hit TP, hit SL, hoặc RSI > overbought, hoặc quá max_hold ngày
    - Mỗi lần chỉ giữ 1 vị thế
    """
    df = df.dropna(subset=['rsi','close']).reset_index(drop=True)
    if len(df) < 60:
        return None

    trades     = []
    equity     = [capital]
    cash       = capital
    in_trade   = False
    entry_price= 0
    entry_idx  = 0
    shares     = 0

    for i in range(1, len(df)):
        row  = df.iloc[i]
        prev = df.iloc[i-1]
        price = row['close']

        if not in_trade:
            # Tín hiệu MUA: RSI cắt lên từ vùng quá bán
            # (hôm qua RSI < oversold, hôm nay RSI >= oversold)
            buy_signal = (
                prev['rsi'] < rsi_oversold and
                row['rsi']  >= rsi_oversold
            )
            if buy_signal and cash > 0:
                # Position sizing: risk_per_trade % vốn / SL%
                risk_amt = cash * risk_per_trade
                shares_float = risk_amt / (price * sl_pct)
                # Làm tròn xuống bội số 100 (lô chẵn VN)
                shares = max(100, int(shares_float / 100) * 100)
                cost   = shares * price * (1 + commission)
                if cost > cash:
                    shares = int(cash / (price * (1 + commission) * 100)) * 100
                    cost   = shares * price * (1 + commission)
                if shares >= 100:
                    cash       -= cost
                    in_trade    = True
                    entry_price = price
                    entry_idx   = i
                    sl_price    = entry_price * (1 - sl_pct)
                    tp_price    = entry_price * (1 + tp_pct)

        else:
            # Điều kiện thoát lệnh
            hold_days   = i - entry_idx
            hit_tp      = row['high'] >= tp_price
            hit_sl      = row['low']  <= sl_price
            rsi_exit    = row['rsi']  > rsi_overbought
            time_exit   = hold_days  >= max_hold

            if hit_tp or hit_sl or rsi_exit or time_exit:
                if hit_sl and not hit_tp:
                    exit_price = sl_price
                    exit_reason= 'SL'
                elif hit_tp:
                    exit_price = tp_price
                    exit_reason= 'TP'
                elif rsi_exit:
                    exit_price = price
                    exit_reason= 'RSI_EXIT'
                else:
                    exit_price = price
                    exit_reason= 'TIME'

                proceeds = shares * exit_price * (1 - commission)
                pnl      = proceeds - shares * entry_price * (1 + commission)
                pnl_pct  = (exit_price - entry_price) / entry_price * 100
                cash    += proceeds
                in_trade = False

                trades.append({
                    'entry_date' : df.iloc[entry_idx]['date'],
                    'exit_date'  : row['date'],
                    'entry_price': entry_price,
                    'exit_price' : exit_price,
                    'shares'     : shares,
                    'pnl'        : pnl,
                    'pnl_pct'    : pnl_pct,
                    'hold_days'  : hold_days,
                    'exit_reason': exit_reason,
                    'won'        : pnl > 0,
                    'rsi_entry'  : df.iloc[entry_idx]['rsi'],
                })

        total_val = cash + (shares * price if in_trade else 0)
        equity.append(total_val)

    if not trades:
        return None

    t = pd.DataFrame(trades)

    # ── Metrics ──────────────────────────────────────────────────────────────
    n          = len(t)
    n_win      = t['won'].sum()
    win_rate   = n_win / n * 100
    avg_win    = t[t['won']]['pnl_pct'].mean() if n_win > 0 else 0
    avg_loss   = t[~t['won']]['pnl_pct'].mean() if (n-n_win) > 0 else 0
    total_pnl  = t['pnl'].sum()
    final_eq   = equity[-1]
    net_pct    = (final_eq - capital) / capital * 100

    # Max Drawdown
    eq_s   = pd.Series(equity)
    peak   = eq_s.cummax()
    dd     = (eq_s - peak) / peak * 100
    max_dd = dd.min()

    # Profit Factor
    gross_profit = t[t['pnl']>0]['pnl'].sum()
    gross_loss   = t[t['pnl']<0]['pnl'].abs().sum()
    profit_factor= gross_profit / gross_loss if gross_loss > 0 else np.inf

    # Sharpe Ratio (annualized, daily returns)
    daily_ret = eq_s.pct_change().dropna()
    sharpe    = (daily_ret.mean() / daily_ret.std() * np.sqrt(252)
                 if daily_ret.std() > 0 else 0)

    # Calmar Ratio
    calmar = net_pct / abs(max_dd) if max_dd != 0 else 0

    # Expectancy (VNĐ trung bình mỗi lệnh)
    expectancy = total_pnl / n

    # t-test: win rate có khác 50% có ý nghĩa thống kê không?
    wins_arr = t['won'].astype(int).values
    t_stat, p_val = stats.ttest_1samp(wins_arr, 0.5)

    return {
        'trades'        : t,
        'equity'        : equity,
        'n_trades'      : n,
        'win_rate'      : win_rate,
        'avg_win_pct'   : avg_win,
        'avg_loss_pct'  : avg_loss,
        'profit_factor' : profit_factor,
        'net_pct'       : net_pct,
        'max_drawdown'  : max_dd,
        'sharpe'        : sharpe,
        'calmar'        : calmar,
        'expectancy'    : expectancy,
        'final_equity'  : final_eq,
        'p_value'       : p_val,
        'significant'   : p_val < 0.05,
        'exit_reasons'  : t['exit_reason'].value_counts().to_dict(),
        'avg_hold_days' : t['hold_days'].mean(),
    }

print('✅ Engine backtest đã sẵn sàng!')


## ▶️ BƯỚC 6 — Chạy Backtest toàn bộ

In [ ]:
print('🚀 Đang chạy backtest...\n')

results = {}
for sym, df in all_data.items():
    r = run_backtest(
        df,
        rsi_oversold   = RSI_OVERSOLD,
        rsi_overbought = RSI_OVERBOUGHT,
        sl_pct         = STOP_LOSS_PCT,
        tp_pct         = TAKE_PROFIT_PCT,
        max_hold       = MAX_HOLD_DAYS,
        commission     = COMMISSION,
        capital        = INITIAL_CAPITAL,
        risk_per_trade = RISK_PER_TRADE,
    )
    if r:
        results[sym] = r
        sig = '✅ *có ý nghĩa*' if r['significant'] else '⚠️ không đủ ý nghĩa thống kê'
        print(f'  {sym:6s}: {r["n_trades"]:3d} lệnh | '
              f'WR={r["win_rate"]:5.1f}% | '
              f'PF={r["profit_factor"]:5.2f} | '
              f'Net={r["net_pct"]:+7.1f}% | '
              f'MDD={r["max_drawdown"]:6.1f}% | p={r["p_value"]:.3f} {sig}')
    else:
        print(f'  {sym:6s}: Không đủ lệnh để phân tích')

print(f'\n✅ Hoàn thành {len(results)}/{len(all_data)} mã')


## 📊 BƯỚC 7 — Bảng kết quả tổng hợp

In [ ]:
if not results:
    print('Không có kết quả để hiển thị')
else:
    summary = pd.DataFrame({
        sym: {
            'Số lệnh'             : r['n_trades'],
            'Win Rate (%)'        : round(r['win_rate'], 1),
            'Profit Factor'       : round(r['profit_factor'], 2),
            'Lợi nhuận ròng (%)'  : round(r['net_pct'], 1),
            'Max Drawdown (%)'    : round(r['max_drawdown'], 1),
            'Sharpe Ratio'        : round(r['sharpe'], 2),
            'Calmar Ratio'        : round(r['calmar'], 2),
            'Avg Win (%)'         : round(r['avg_win_pct'], 1),
            'Avg Loss (%)'        : round(r['avg_loss_pct'], 1),
            'Expectancy (VNĐ)'    : f"{r['expectancy']:,.0f}",
            'Avg Hold (ngày)'     : round(r['avg_hold_days'], 1),
            'p-value'             : round(r['p_value'], 3),
            'Ý nghĩa thống kê'    : '✅ Có' if r['significant'] else '❌ Không',
        }
        for sym, r in results.items()
    }).T

    # Style bảng
    def color_val(val):
        try:
            v = float(str(val).replace(',',''))
            if v > 0:  return 'color: #26a69a; font-weight: bold'
            if v < 0:  return 'color: #ef5350; font-weight: bold'
        except: pass
        return ''

    styled = summary.style.applymap(color_val, subset=[
        'Lợi nhuận ròng (%)','Win Rate (%)','Profit Factor',
        'Max Drawdown (%)','Sharpe Ratio'
    ]).set_properties(**{'text-align':'center'})

    print(f'\n📋 KẾT QUẢ BACKTEST RSI REVERSAL — {YEARS} NĂM DỮ LIỆU THẬT')
    print(f'   SL={STOP_LOSS_PCT*100:.0f}% | TP={TAKE_PROFIT_PCT*100:.0f}% | '
          f'RSI mua<{RSI_OVERSOLD} | RSI bán>{RSI_OVERBOUGHT} | Phí={COMMISSION*100:.2f}%/chiều\n')
    display(styled)

    # Tóm tắt tổng
    avg_wr  = np.mean([r['win_rate']  for r in results.values()])
    avg_net = np.mean([r['net_pct']   for r in results.values()])
    avg_pf  = np.mean([r['profit_factor'] for r in results.values() if r['profit_factor'] != np.inf])
    n_sig   = sum(1 for r in results.values() if r['significant'])

    print(f'\n📌 TRUNG BÌNH TOÀN BỘ:')
    print(f'   Win Rate TB    : {avg_wr:.1f}%')
    print(f'   Lợi nhuận TB   : {avg_net:+.1f}%')
    print(f'   Profit Factor TB: {avg_pf:.2f}')
    print(f'   Có ý nghĩa TK  : {n_sig}/{len(results)} mã (p < 0.05)')
    print()
    if avg_wr > 55 and avg_pf > 1.3:
        print('  🟢 Chiến lược có edge dương trên tập dữ liệu này')
    elif avg_wr > 50:
        print('  🟡 Chiến lược nhỉnh hơn random nhưng không rõ ràng')
    else:
        print('  🔴 Chiến lược KHÔNG có edge rõ ràng trên dữ liệu này')
    print('  ⚠️  Đây là kết quả trên dữ liệu QUÁ KHỨ, không đảm bảo tương lai!')


## 📈 BƯỚC 8 — Biểu đồ Equity Curve & Phân tích

In [ ]:
if results:
    n_plots = len(results)
    cols    = min(3, n_plots)
    rows    = (n_plots + cols - 1) // cols

    fig = plt.figure(figsize=(6*cols, 4*rows))
    fig.patch.set_facecolor('#1a1a2e')

    for idx, (sym, r) in enumerate(results.items(), 1):
        ax  = fig.add_subplot(rows, cols, idx)
        ax.set_facecolor('#16213e')

        eq  = pd.Series(r['equity'])
        pct = (eq / eq.iloc[0] - 1) * 100
        color = CLR['up'] if pct.iloc[-1] >= 0 else CLR['down']

        ax.plot(pct.values, color=color, lw=1.5)
        ax.fill_between(range(len(pct)), pct.values, 0,
                        where=pct.values>=0, alpha=0.15, color=CLR['up'])
        ax.fill_between(range(len(pct)), pct.values, 0,
                        where=pct.values<0,  alpha=0.15, color=CLR['down'])
        ax.axhline(0, color='white', lw=0.5, alpha=0.4)

        sig_mark = '✅' if r['significant'] else '⚠️'
        ax.set_title(
            f"{sym}  {sig_mark}\n"
            f"WR={r['win_rate']:.1f}%  PF={r['profit_factor']:.2f}  Net={r['net_pct']:+.1f}%",
            color='white', fontsize=9
        )
        ax.tick_params(colors='gray', labelsize=7)
        ax.set_ylabel('Lợi nhuận (%)', color='gray', fontsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor('#333355')

    fig.suptitle(
        f'Equity Curves — RSI Reversal ({YEARS} năm dữ liệu thực TCBS)',
        color='white', fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.savefig('equity_curves.png', dpi=120, bbox_inches='tight',
                facecolor='#1a1a2e')
    plt.show()
    print('💾 Đã lưu: equity_curves.png')


## 🔬 BƯỚC 9 — Walk-Forward Testing (Kiểm tra tính bền vững)

In [ ]:
def walk_forward_test(
    df: pd.DataFrame,
    symbol: str,
    n_splits: int = 6,
    train_ratio: float = 0.6,   # 60% dữ liệu để tối ưu tham số
    test_ratio: float = 0.4,    # 40% để kiểm tra ngoài mẫu
) -> pd.DataFrame:
    """
    Walk-Forward Analysis:
    - Chia dữ liệu thành N cửa sổ thời gian
    - Mỗi cửa sổ: dùng phần TRAIN để tìm tham số RSI tốt nhất
    - Áp dụng tham số đó vào phần TEST (dữ liệu chưa thấy)
    - Kết quả TEST mới là con số thực sự đáng tin
    """
    df = df.dropna(subset=['rsi','close']).reset_index(drop=True)
    total = len(df)
    window_size = total // n_splits

    # Lưới tham số để tối ưu trên tập TRAIN
    param_grid = list(itertools.product(
        [25, 30, 35],   # RSI oversold
        [65, 70, 75],   # RSI overbought
    ))

    wf_results = []

    for fold in range(n_splits - 1):
        train_start = fold * window_size
        train_end   = train_start + int(window_size * (n_splits-1) / n_splits)
        test_start  = train_end
        test_end    = min(test_start + window_size, total)

        df_train = df.iloc[train_start:train_end]
        df_test  = df.iloc[test_start:test_end]

        if len(df_train) < 60 or len(df_test) < 20:
            continue

        # ── Tối ưu tham số trên TRAIN ──────────────────────────────────────
        best_pf    = -np.inf
        best_params= (30, 70)

        for (os, ob) in param_grid:
            r_train = run_backtest(
                df_train, rsi_oversold=os, rsi_overbought=ob,
                sl_pct=STOP_LOSS_PCT, tp_pct=TAKE_PROFIT_PCT,
                max_hold=MAX_HOLD_DAYS, commission=COMMISSION,
                capital=INITIAL_CAPITAL,
            )
            if r_train and r_train['profit_factor'] != np.inf:
                if r_train['profit_factor'] > best_pf:
                    best_pf     = r_train['profit_factor']
                    best_params = (os, ob)

        # ── Áp dụng tham số tốt nhất vào TEST ─────────────────────────────
        r_test = run_backtest(
            df_test,
            rsi_oversold   = best_params[0],
            rsi_overbought = best_params[1],
            sl_pct=STOP_LOSS_PCT, tp_pct=TAKE_PROFIT_PCT,
            max_hold=MAX_HOLD_DAYS, commission=COMMISSION,
            capital=INITIAL_CAPITAL,
        )

        train_period = (
            f"{df_train['date'].iloc[0].strftime('%m/%Y')}–"
            f"{df_train['date'].iloc[-1].strftime('%m/%Y')}"
        )
        test_period = (
            f"{df_test['date'].iloc[0].strftime('%m/%Y')}–"
            f"{df_test['date'].iloc[-1].strftime('%m/%Y')}"
        )

        row = {
            'Fold'              : fold + 1,
            'Train period'      : train_period,
            'Best RSI OS/OB'    : f"{best_params[0]}/{best_params[1]}",
            'Train PF'          : round(best_pf, 2),
            'Test period'       : test_period,
            'Test lệnh'         : r_test['n_trades']   if r_test else 0,
            'Test WR (%)'       : round(r_test['win_rate'],   1) if r_test else 0,
            'Test PF'           : round(r_test['profit_factor'], 2) if r_test and r_test['profit_factor'] != np.inf else 0,
            'Test Net (%)'      : round(r_test['net_pct'],    1) if r_test else 0,
            'Test MDD (%)'      : round(r_test['max_drawdown'], 1) if r_test else 0,
            'Degradation'       : round(best_pf - (r_test['profit_factor'] if r_test and r_test['profit_factor'] != np.inf else 0), 2),
        }
        wf_results.append(row)

    return pd.DataFrame(wf_results)


# Chạy walk-forward cho từng mã
print('🔬 Đang chạy Walk-Forward Testing...')
print('   (Mỗi mã tối ưu 9 tổ hợp tham số × N folds — mất 1-2 phút)\n')

wf_all = {}
for sym, df in all_data.items():
    print(f'  {sym}...', end=' ')
    wf = walk_forward_test(df, sym, n_splits=WF_N_SPLITS)
    if not wf.empty:
        wf_all[sym] = wf
        avg_test_pf = wf['Test PF'].mean()
        avg_test_wr = wf['Test WR (%)'].mean()
        print(f'Avg Test PF={avg_test_pf:.2f} | Avg Test WR={avg_test_wr:.1f}%')
    else:
        print('Không đủ dữ liệu')

print('\n✅ Walk-Forward hoàn tất!')


## 📋 BƯỚC 10 — Kết quả Walk-Forward chi tiết

In [ ]:
for sym, wf in wf_all.items():
    print(f'\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    print(f'  {sym} — Walk-Forward ({WF_N_SPLITS} folds)')
    print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

    def style_wf(val):
        try:
            v = float(val)
            if v > 0: return 'color: #26a69a'
            if v < 0: return 'color: #ef5350'
        except: pass
        return ''

    display(wf.style.applymap(style_wf, subset=['Test Net (%)','Test PF','Degradation']))

    # Đánh giá tính bền vững
    n_positive = (wf['Test Net (%)'] > 0).sum()
    pct_pos    = n_positive / len(wf) * 100
    avg_degrad = wf['Degradation'].mean()

    print(f'  📊 Tóm tắt {sym}:')
    print(f'     Folds có lợi nhuận dương: {n_positive}/{len(wf)} ({pct_pos:.0f}%)')
    print(f'     Degradation TB (Train→Test PF): {avg_degrad:.2f}')

    if pct_pos >= 67 and avg_degrad < 0.8:
        verdict = '🟢 BỀN VỮNG — Chiến lược có khả năng hoạt động ngoài mẫu'
    elif pct_pos >= 50:
        verdict = '🟡 TRUNG BÌNH — Một số giai đoạn hoạt động, một số không'
    else:
        verdict = '🔴 KHÔNG BỀN VỮNG — Overfit trên dữ liệu huấn luyện'
    print(f'     Đánh giá: {verdict}')


## 📉 BƯỚC 11 — Biểu đồ Walk-Forward tổng hợp

In [ ]:
if wf_all:
    syms = list(wf_all.keys())
    n    = len(syms)
    cols = min(3, n)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(7*cols, 4*rows))
    fig.patch.set_facecolor('#1a1a2e')
    axes_flat = np.array(axes).flatten() if n > 1 else [axes]

    for idx, sym in enumerate(syms):
        ax  = axes_flat[idx]
        wf  = wf_all[sym]
        ax.set_facecolor('#16213e')

        folds       = wf['Fold'].values
        train_pf    = wf['Train PF'].values
        test_pf     = wf['Test PF'].values
        test_net    = wf['Test Net (%)'].values

        x = np.arange(len(folds))
        w = 0.35
        b1 = ax.bar(x - w/2, train_pf, w, label='Train PF',
                    color=CLR['up'], alpha=0.7)
        b2 = ax.bar(x + w/2, test_pf,  w, label='Test PF',
                    color=CLR['down'], alpha=0.7)

        ax2 = ax.twinx()
        ax2.plot(x, test_net, 'o--', color=CLR['gold'],
                 lw=1.5, ms=6, label='Test Net%')
        ax2.axhline(0, color='white', lw=0.5, alpha=0.3)
        ax2.tick_params(colors='gray', labelsize=7)
        ax2.set_ylabel('Test Net (%)', color=CLR['gold'], fontsize=8)

        ax.axhline(1, color='white', lw=0.8, ls='--', alpha=0.4)
        ax.set_xticks(x)
        ax.set_xticklabels([f'F{f}' for f in folds], color='gray', fontsize=8)
        ax.tick_params(colors='gray', labelsize=7)
        ax.set_ylabel('Profit Factor', color='gray', fontsize=8)
        ax.legend(loc='upper left', fontsize=7, labelcolor='white',
                  facecolor='#1a1a2e', edgecolor='gray')

        pct_pos = (test_net > 0).mean() * 100
        color_t = CLR['green'] if pct_pos >= 60 else (CLR['gold'] if pct_pos >= 40 else CLR['red'])
        ax.set_title(f'{sym}  —  {pct_pos:.0f}% folds dương',
                     color=color_t, fontsize=10, fontweight='bold')
        for spine in ax.spines.values():
            spine.set_edgecolor('#333355')

    for idx in range(len(syms), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.suptitle(
        'Walk-Forward Testing — Train PF vs Test PF (cột) & Test Net% (đường vàng)',
        color='white', fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('walkforward.png', dpi=120, bbox_inches='tight',
                facecolor='#1a1a2e')
    plt.show()
    print('💾 Đã lưu: walkforward.png')


## 🎛️ BƯỚC 12 — Tối ưu tham số RSI (Optimization Heatmap)

In [ ]:
# Chọn 1 mã để vẽ heatmap tối ưu tham số
OPTIMIZE_SYMBOL = list(all_data.keys())[0]   # ← Đổi tên mã ở đây nếu muốn

print(f'🎛️  Tối ưu tham số RSI cho {OPTIMIZE_SYMBOL}...')
df_opt = all_data[OPTIMIZE_SYMBOL]

os_range = [20, 25, 30, 35, 40]
ob_range = [60, 65, 70, 75, 80]

heatmap_wr = np.zeros((len(os_range), len(ob_range)))
heatmap_pf = np.zeros((len(os_range), len(ob_range)))
heatmap_net= np.zeros((len(os_range), len(ob_range)))

for i, os in enumerate(os_range):
    for j, ob in enumerate(ob_range):
        if os >= ob:
            heatmap_wr[i,j] = np.nan
            heatmap_pf[i,j] = np.nan
            heatmap_net[i,j]= np.nan
            continue
        r = run_backtest(df_opt, rsi_oversold=os, rsi_overbought=ob,
                         sl_pct=STOP_LOSS_PCT, tp_pct=TAKE_PROFIT_PCT,
                         max_hold=MAX_HOLD_DAYS, commission=COMMISSION)
        if r:
            heatmap_wr[i,j]  = r['win_rate']
            heatmap_pf[i,j]  = min(r['profit_factor'], 5) if r['profit_factor'] != np.inf else 5
            heatmap_net[i,j] = r['net_pct']
        else:
            heatmap_wr[i,j] = heatmap_pf[i,j] = heatmap_net[i,j] = 0

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#1a1a2e')

titles  = ['Win Rate (%)', 'Profit Factor', 'Net Return (%)']
data_m  = [heatmap_wr, heatmap_pf, heatmap_net]
cmaps   = ['RdYlGn', 'RdYlGn', 'RdYlGn']

for ax, title, data, cmap in zip(axes, titles, data_m, cmaps):
    ax.set_facecolor('#16213e')
    sns.heatmap(
        data, ax=ax, cmap=cmap, annot=True, fmt='.1f',
        xticklabels=ob_range, yticklabels=os_range,
        linewidths=0.5, linecolor='#333355',
        annot_kws={'size': 9},
        cbar_kws={'shrink': 0.8}
    )
    ax.set_title(f'{OPTIMIZE_SYMBOL} — {title}',
                 color='white', fontsize=11, fontweight='bold')
    ax.set_xlabel('RSI Overbought', color='gray')
    ax.set_ylabel('RSI Oversold',   color='gray')
    ax.tick_params(colors='gray')

fig.suptitle(
    f'Optimization Heatmap — {OPTIMIZE_SYMBOL}  (Xanh = tốt hơn)',
    color='white', fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('heatmap_optimization.png', dpi=120, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print('💾 Đã lưu: heatmap_optimization.png')
print()
print('⚠️  Lưu ý: Các vùng màu xanh trên heatmap là kết quả TỐT NHẤT trên dữ liệu QUÁ KHỨ.')
print('   KHÔNG chọn tham số dựa hoàn toàn vào đây — đó là data snooping bias!')
print('   Hãy dùng Walk-Forward (bước 9-10) để xác nhận tham số thực sự bền vững.')


## 📑 BƯỚC 13 — Báo cáo tổng kết & Đánh giá trung thực

In [ ]:
print('=' * 65)
print('  BÁO CÁO TỔNG KẾT — RSI REVERSAL BACKTEST')
print(f'  Dữ liệu: {YEARS} năm thực từ TCBS | {len(results)} cổ phiếu HOSE/HNX')
print('=' * 65)

# ── Tổng hợp backtest ────────────────────────────────────────────────────────
print('\n📊 1. KẾT QUẢ BACKTEST (trên toàn bộ dữ liệu):')
for sym, r in results.items():
    icon = '🟢' if r['net_pct'] > 5 and r['profit_factor'] > 1.3 else (
           '🟡' if r['net_pct'] > 0 else '🔴')
    print(f'  {icon} {sym:5s}: Net={r["net_pct"]:+6.1f}% | '
          f'WR={r["win_rate"]:5.1f}% | '
          f'PF={r["profit_factor"]:4.2f} | '
          f'MDD={r["max_drawdown"]:5.1f}% | '
          f'n={r["n_trades"]:3d} lệnh | '
          f'p={r["p_value"]:.3f}')

# ── Tổng hợp walk-forward ────────────────────────────────────────────────────
print('\n🔬 2. KẾT QUẢ WALK-FORWARD (ngoài mẫu — đáng tin hơn):')
robust_symbols = []
for sym, wf in wf_all.items():
    pct_pos    = (wf['Test Net (%)'] > 0).mean() * 100
    avg_test_pf= wf['Test PF'].mean()
    icon = '🟢' if pct_pos >= 67 else ('🟡' if pct_pos >= 50 else '🔴')
    if pct_pos >= 67: robust_symbols.append(sym)
    print(f'  {icon} {sym:5s}: {pct_pos:.0f}% folds dương | '
          f'Avg Test PF={avg_test_pf:.2f}')

# ── Đánh giá tổng quan ───────────────────────────────────────────────────────
print('\n⚖️  3. ĐÁNH GIÁ KHÁCH QUAN:')
all_wrs  = [r['win_rate']  for r in results.values()]
all_nets = [r['net_pct']   for r in results.values()]
all_pfs  = [r['profit_factor'] for r in results.values() if r['profit_factor'] != np.inf]
n_sig    = sum(1 for r in results.values() if r['significant'])

print(f'  Win Rate TB       : {np.mean(all_wrs):.1f}%  (random = 50%)')
print(f'  Net Return TB     : {np.mean(all_nets):+.1f}%')
print(f'  Profit Factor TB  : {np.mean(all_pfs):.2f}  (> 1.0 = có edge)')
print(f'  Ý nghĩa thống kê  : {n_sig}/{len(results)} mã (p < 0.05)')
print(f'  Mã bền vững (WF)  : {robust_symbols if robust_symbols else "Chưa có"}')

print('\n⚠️  4. GIỚI HẠN CẦN BIẾT:')
limits = [
    'Look-ahead bias: code đã cố gắng tránh nhưng không thể loại bỏ 100%',
    'Slippage: giả định mua/bán đúng giá đóng cửa — thực tế sẽ khác',
    'Thanh khoản: với vốn lớn, lệnh có thể không khớp đủ số lượng',
    'Regime change: thị trường 2022-2024 có thể không giống 2025-2026',
    'Phí thực tế: thuế 0.1% bán chưa tính — tổng phí thực ~0.35%/lệnh',
    'Dữ liệu TCBS: chất lượng tốt nhưng có thể có lỗi ở một số mã nhỏ',
]
for l in limits:
    print(f'  • {l}')

print('\n💡 5. KHUYẾN NGHỊ:')
if robust_symbols:
    print(f'  Các mã có tín hiệu bền vững nhất: {robust_symbols}')
    print('  → Chỉ trade các mã này, với vốn nhỏ trước để xác nhận thực tế')
else:
    print('  → Chiến lược RSI thuần chưa đủ bền vững trên dữ liệu này')
    print('  → Thử kết hợp thêm: Volume filter, Trend filter (SMA50), hoặc MACD confirm')
print('  → KHÔNG trade với toàn bộ vốn dựa trên backtest')
print('  → Paper trade (theo dõi tín hiệu không bỏ tiền thật) ít nhất 2-3 tháng')
print('  → Kết quả backtest là lịch sử, KHÔNG đảm bảo tương lai')
print()
print('=' * 65)


## 💾 BƯỚC 14 — Xuất kết quả ra file Excel
Tải về máy/điện thoại để xem chi tiết từng lệnh

In [ ]:
from google.colab import files

with pd.ExcelWriter('vn_backtest_results.xlsx', engine='openpyxl') as writer:

    # Sheet 1: Tóm tắt
    if results:
        summary_df = pd.DataFrame({
            sym: {
                'Số lệnh'        : r['n_trades'],
                'Win Rate (%)'   : round(r['win_rate'],1),
                'Profit Factor'  : round(r['profit_factor'],2),
                'Net Return (%)'  : round(r['net_pct'],1),
                'Max DD (%)'     : round(r['max_drawdown'],1),
                'Sharpe'         : round(r['sharpe'],2),
                'Avg Win (%)'    : round(r['avg_win_pct'],1),
                'Avg Loss (%)'   : round(r['avg_loss_pct'],1),
                'Expectancy'     : round(r['expectancy'],0),
                'p-value'        : round(r['p_value'],4),
                'Ý nghĩa TK'     : 'Có' if r['significant'] else 'Không',
            } for sym, r in results.items()
        }).T
        summary_df.to_excel(writer, sheet_name='Tóm tắt')

    # Sheet 2-N: Chi tiết từng lệnh theo mã
    for sym, r in results.items():
        t = r['trades'][['entry_date','exit_date','entry_price',
                          'exit_price','shares','pnl','pnl_pct',
                          'hold_days','exit_reason','won','rsi_entry']].copy()
        t['pnl']       = t['pnl'].round(0)
        t['pnl_pct']   = t['pnl_pct'].round(2)
        t['rsi_entry'] = t['rsi_entry'].round(1)
        t.to_excel(writer, sheet_name=f'{sym}_trades', index=False)

    # Sheet Walk-Forward
    for sym, wf in wf_all.items():
        wf.to_excel(writer, sheet_name=f'{sym}_WalkFwd', index=False)

print('✅ Đã tạo file Excel!')
files.download('vn_backtest_results.xlsx')
print('📥 File đang được tải về thiết bị của bạn...')
